# Activity No. 2: Classification using SVM, Decision Tree, KNN and Naïve Bayes

| | |
|---|---|
| **Name** | Yash Galande |
| **PRN** | 202401040179 |
| **Batch** | A2 |
| **Roll No.** | 41 |
| **Datasets** | Breast Cancer Wisconsin, Wine, Digits (built into scikit-learn) |
| **GitHub** | [View repository](https://github.com/PASTE-YOUR-LINK-HERE) |

## Aim
To develop classification models using **Support Vector Machine (SVM), Decision Tree, K-Nearest Neighbours (KNN) and Naïve Bayes**, evaluate them using accuracy, precision, recall, F1-score and loss, and summarise a comparative analysis discussing the suitability of each algorithm for different classification tasks.

## Problem Statement
No single classifier is best for every problem. Different algorithms make different assumptions and behave differently on binary vs. multiclass problems, small vs. high-dimensional data, and noisy vs. clean data. We therefore train all four algorithms on **three different tasks** and compare them.

| Task | Dataset | Type | Samples | Features | Classes |
|---|---|---|---|---|---|
| Medical diagnosis | Breast Cancer | Binary, tabular | 569 | 30 | 2 |
| Product/chemistry classification | Wine | Multiclass, small tabular | 178 | 13 | 3 |
| Image recognition | Digits (8×8 images) | Multiclass, high-dimensional | 1797 | 64 | 10 |

## Objectives
1. Load and explore the datasets.
2. Split into train/test sets (stratified) and scale features where required.
3. Train SVM, Decision Tree, KNN and Gaussian Naïve Bayes.
4. Evaluate with accuracy, precision, recall, F1-score, **log loss** (and hinge loss for SVM).
5. Visualise confusion matrices, ROC curves and decision behaviour.
6. Cross-validate and tune hyper-parameters.
7. Compare algorithms across tasks and discuss their suitability.

## Theory
* **SVM** – finds the maximum-margin hyperplane; with the RBF kernel it handles non-linear boundaries. Needs feature scaling. Training is costly on very large datasets.
* **Decision Tree** – learns if-then splits; easy to interpret, needs no scaling, but can overfit.
* **KNN** – lazy learner; classifies by majority vote of the *k* nearest points. Sensitive to scaling and to high dimensions; slow at prediction time.
* **Naïve Bayes** – applies Bayes' theorem assuming features are conditionally independent (Gaussian NB assumes normal feature distributions). Very fast, works with little data, but the independence assumption is often violated.

**Metrics**
* *Accuracy* = correct predictions / total. *Precision* = TP/(TP+FP). *Recall* = TP/(TP+FN). *F1* = harmonic mean of precision and recall.
* *Log loss (cross-entropy)* penalises confident wrong predictions: lower is better. It needs predicted probabilities. *Hinge loss* is the margin-based loss used by SVMs.

## GitHub Repository
🔗 **Source code:** [https://github.com/PASTE-YOUR-LINK-HERE](https://github.com/PASTE-YOUR-LINK-HERE)

## 1. Import libraries

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss, hinge_loss,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_curve, auc, roc_auc_score)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load and explore the datasets

In [ ]:
datasets = {
    "Breast Cancer": load_breast_cancer(),
    "Wine": load_wine(),
    "Digits": load_digits(),
}

summary = []
for name, d in datasets.items():
    summary.append({"Dataset": name, "Samples": d.data.shape[0], "Features": d.data.shape[1],
                    "Classes": len(np.unique(d.target)),
                    "Missing values": int(np.isnan(d.data).sum())})
display(pd.DataFrame(summary))

# Look at the primary dataset (Breast Cancer)
bc = datasets["Breast Cancer"]
bc_df = pd.DataFrame(bc.data, columns=bc.feature_names)
bc_df["target"] = bc.target
print("Breast Cancer class counts (0 = malignant, 1 = benign):")
display(bc_df["target"].value_counts())
display(bc_df.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, d) in zip(axes, datasets.items()):
    vals, cnts = np.unique(d.target, return_counts=True)
    ax.bar(vals.astype(str), cnts, color="#4c72b0")
    ax.set(title=f"{name}: class distribution", xlabel="Class", ylabel="Count")
plt.tight_layout(); plt.show()

# A few sample digit images
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
dg = datasets["Digits"]
for ax, img, lab in zip(axes, dg.images[:8], dg.target[:8]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(lab); ax.axis("off")
plt.suptitle("Sample digit images"); plt.show()

## 3. Define the models
SVM and KNN are distance/margin based, so they are wrapped in a pipeline with `StandardScaler`. Decision Tree and Naïve Bayes do not need scaling. `probability=True` lets SVM output probabilities so that log loss can be computed. The tree depth is limited (`max_depth=5`) to reduce overfitting and extreme probabilities.

In [ ]:
def get_models():
    return {
        "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale",
                                                         probability=True, random_state=RANDOM_STATE)),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "Naive Bayes": GaussianNB(),
    }

def evaluate(model, X_tr, X_te, y_tr, y_te):
    """Train a model and return metrics, predictions and probabilities."""
    t0 = time.perf_counter(); model.fit(X_tr, y_tr); fit_t = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(X_te); pred_t = time.perf_counter() - t0
    proba = model.predict_proba(X_te)
    classes = np.unique(y_tr)
    return {
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_te, pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_te, pred, average="weighted", zero_division=0),
        "Log Loss": log_loss(y_te, proba, labels=classes),
        "Train Time (s)": fit_t, "Predict Time (s)": pred_t,
    }, pred, proba

## 4. Task 1 – Breast Cancer diagnosis (binary classification)

In [ ]:
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

bc_results, bc_models, bc_preds, bc_probas = [], {}, {}, {}
for name, model in get_models().items():
    m, pred, proba = evaluate(model, X_train, X_test, y_train, y_test)
    bc_models[name], bc_preds[name], bc_probas[name] = model, pred, proba
    bc_results.append({"Model": name, **m})

bc_df_res = pd.DataFrame(bc_results).sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(bc_df_res.round(4))

In [ ]:
# Hinge loss for SVM (margin-based loss, uses decision_function)
svm_pipe = bc_models["SVM (RBF)"]
print("SVM hinge loss:", round(hinge_loss(y_test, svm_pipe.decision_function(X_test)), 4))

# Metric comparison chart
bc_df_res.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0.8, 1.0))
plt.title("Breast Cancer – Performance Metrics"); plt.ylabel("Score"); plt.xticks(rotation=15, ha="right")
plt.tight_layout(); plt.show()

bc_df_res.set_index("Model")["Log Loss"].plot(kind="bar", figsize=(7, 4), color="#c44e52")
plt.title("Breast Cancer – Log Loss (lower is better)"); plt.ylabel("Log loss"); plt.xticks(rotation=15, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, pred) in zip(axes, bc_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=bc.target_names).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

# Classification report of the best model
best = bc_df_res.loc[0, "Model"]
print("Best model by F1:", best)
print(classification_report(y_test, bc_preds[best], target_names=bc.target_names))

# ROC curves
plt.figure(figsize=(7, 6))
for name, proba in bc_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves – Breast Cancer"); plt.legend(loc="lower right"); plt.show()

In [ ]:
# Interpretability: the decision tree's learned rules
plt.figure(figsize=(18, 7))
plot_tree(bc_models["Decision Tree"], feature_names=bc.feature_names, class_names=bc.target_names,
          filled=True, max_depth=3, fontsize=8)
plt.title("Decision Tree (top 3 levels)"); plt.show()

**Note on medical tasks:** in cancer screening a *false negative* (missing a malignant tumour) is much costlier than a false positive, so **recall for the malignant class** matters more than raw accuracy.

## 5. Task 2 – Wine classification (small multiclass dataset)

In [ ]:
def run_task(name, data):
    X, y = data.data, data.target
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    rows, models, preds = [], {}, {}
    for mname, model in get_models().items():
        m, pred, _ = evaluate(model, X_tr, X_te, y_tr, y_te)
        models[mname], preds[mname] = model, pred
        rows.append({"Model": mname, **m})
    res = pd.DataFrame(rows).sort_values("F1 Score", ascending=False).reset_index(drop=True)
    return res, models, preds, (X_tr, X_te, y_tr, y_te)

wine_res, wine_models, wine_preds, wine_split = run_task("Wine", datasets["Wine"])
display(wine_res.round(4))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (n, p) in zip(axes, wine_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(wine_split[3], p)).plot(ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.show()

## 6. Task 3 – Digit recognition (high-dimensional multiclass)

In [ ]:
dig_res, dig_models, dig_preds, dig_split = run_task("Digits", datasets["Digits"])
display(dig_res.round(4))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (n, p) in zip(axes, dig_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(dig_split[3], p)).plot(ax=ax, cmap="Oranges", colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.show()

# Show some misclassified digits for the weakest model
worst = dig_res.iloc[-1]["Model"]
X_te, y_te = dig_split[1], dig_split[3]
wrong = np.where(dig_preds[worst] != y_te)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.6 * len(wrong), 2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(X_te[i].reshape(8, 8), cmap="gray_r"); ax.axis("off")
    ax.set_title(f"T:{y_te[i]} P:{dig_preds[worst][i]}", fontsize=9)
plt.suptitle(f"Misclassified digits – {worst}"); plt.show()

## 7. Cross-validation (more reliable than a single split)
5-fold stratified CV gives mean ± standard deviation, reducing the dependence on one lucky/unlucky split.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for dname, d in datasets.items():
    for mname, model in get_models().items():
        s = cross_val_score(model, d.data, d.target, cv=skf, scoring="accuracy")
        cv_rows.append({"Dataset": dname, "Model": mname, "CV Mean Acc": s.mean(), "CV Std": s.std()})
cv_df = pd.DataFrame(cv_rows)
cv_pivot = cv_df.pivot(index="Model", columns="Dataset", values="CV Mean Acc")
display(cv_pivot.round(4))

cv_pivot.plot(kind="bar", figsize=(10, 5), ylim=(0.5, 1.02))
plt.title("5-Fold CV Accuracy across tasks"); plt.ylabel("Accuracy"); plt.xticks(rotation=15, ha="right")
plt.tight_layout(); plt.show()

## 8. Hyper-parameter tuning (Breast Cancer)

In [ ]:
param_grids = {
    "SVM (RBF)": (make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
                  {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.001]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                      {"max_depth": [2, 3, 4, 5, 7, None], "min_samples_leaf": [1, 3, 5]}),
    "KNN": (make_pipeline(StandardScaler(), KNeighborsClassifier()),
            {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11],
             "kneighborsclassifier__weights": ["uniform", "distance"]}),
    "Naive Bayes": (GaussianNB(), {"var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5, 1e-3]}),
}
tuned = []
for name, (est, grid) in param_grids.items():
    gs = GridSearchCV(est, grid, cv=skf, scoring="f1_weighted").fit(X_train, y_train)
    tuned.append({"Model": name, "Best Params": gs.best_params_, "CV F1": gs.best_score_,
                  "Test Accuracy": accuracy_score(y_test, gs.predict(X_test))})
display(pd.DataFrame(tuned).round(4))

In [ ]:
# Effect of k on KNN (bias-variance trade-off)
ks = range(1, 31)
tr_acc, te_acc = [], []
for k in ks:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    tr_acc.append(m.score(X_train, y_train)); te_acc.append(m.score(X_test, y_test))
plt.figure(figsize=(7, 4)); plt.plot(ks, tr_acc, label="Train"); plt.plot(ks, te_acc, label="Test")
plt.xlabel("k (neighbours)"); plt.ylabel("Accuracy"); plt.title("KNN: effect of k"); plt.legend(); plt.show()

# Effect of tree depth (overfitting)
depths = range(1, 16)
tr_acc, te_acc = [], []
for dpt in depths:
    m = DecisionTreeClassifier(max_depth=dpt, random_state=RANDOM_STATE).fit(X_train, y_train)
    tr_acc.append(m.score(X_train, y_train)); te_acc.append(m.score(X_test, y_test))
plt.figure(figsize=(7, 4)); plt.plot(depths, tr_acc, label="Train"); plt.plot(depths, te_acc, label="Test")
plt.xlabel("max_depth"); plt.ylabel("Accuracy"); plt.title("Decision Tree: effect of depth"); plt.legend(); plt.show()

## 9. Overall comparative analysis

In [ ]:
all_res = []
for dname, res in [("Breast Cancer", bc_df_res), ("Wine", wine_res), ("Digits", dig_res)]:
    r = res.copy(); r.insert(0, "Dataset", dname); all_res.append(r)
all_res = pd.concat(all_res)

for metric in ["F1 Score", "Log Loss", "Train Time (s)", "Predict Time (s)"]:
    print(f"\n=== {metric} ===")
    display(all_res.pivot(index="Model", columns="Dataset", values=metric).round(4))

# Average rank across tasks (1 = best); for Log Loss lower is better
rank_f1 = all_res.groupby("Dataset")["F1 Score"].rank(ascending=False)
rank_ll = all_res.groupby("Dataset")["Log Loss"].rank(ascending=True)
all_res["Rank F1"], all_res["Rank LogLoss"] = rank_f1, rank_ll
avg_rank = all_res.groupby("Model")[["Rank F1", "Rank LogLoss"]].mean().sort_values(["Rank F1", "Rank LogLoss"])
display(avg_rank.round(2))
avg_rank.plot(kind="bar", figsize=(8, 4)); plt.ylabel("Average rank (1 = best)")
plt.title("Average rank across the three tasks"); plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

## 10. Suitability of each algorithm

| Algorithm | Strengths | Weaknesses | Best suited for |
|---|---|---|---|
| **SVM (RBF)** | High accuracy; effective in high dimensions; handles non-linear boundaries; robust margin | Needs scaling; slow on very large data; harder to interpret; sensitive to *C*/*γ* | Image/text classification, bioinformatics, small-to-medium datasets with many features |
| **Decision Tree** | Highly interpretable; no scaling needed; handles mixed feature types; captures non-linearity | Overfits easily; unstable; usually less accurate alone; poorly calibrated probabilities (high log loss) | Tasks that need explainable rules – credit approval, medical triage, business rules |
| **KNN** | Simple, no training phase; flexible decision boundary; good with clean, well-scaled data | Slow prediction on large data; hurt by irrelevant features and high dimensions; needs scaling | Small datasets, recommendation-style problems, baselines |
| **Naïve Bayes** | Very fast; works with little data; gives probabilities; scales well | Independence assumption rarely true; limited accuracy when features are correlated | Text/spam filtering, real-time prediction, quick baselines, very high-dimensional sparse data |

In [ ]:
best_bc, best_wine, best_dig = bc_df_res.iloc[0], wine_res.iloc[0], dig_res.iloc[0]
fastest = all_res.groupby("Model")["Train Time (s)"].mean().idxmin()
lowest_ll = all_res.groupby("Model")["Log Loss"].mean().idxmin()
display(Markdown(f'''
### Findings from this run
* **Breast Cancer (binary):** best model was **{best_bc["Model"]}** (accuracy {best_bc["Accuracy"]:.3f}, F1 {best_bc["F1 Score"]:.3f}, log loss {best_bc["Log Loss"]:.3f}).
* **Wine (small multiclass):** best model was **{best_wine["Model"]}** (accuracy {best_wine["Accuracy"]:.3f}, F1 {best_wine["F1 Score"]:.3f}).
* **Digits (high-dimensional multiclass):** best model was **{best_dig["Model"]}** (accuracy {best_dig["Accuracy"]:.3f}, F1 {best_dig["F1 Score"]:.3f}).
* **Best average rank (F1):** **{avg_rank.index[0]}**; **lowest average log loss:** **{lowest_ll}**; **fastest to train:** **{fastest}**.
'''))

## 11. Conclusion
* All four algorithms were implemented, evaluated with accuracy, precision, recall, F1-score and log loss, and tested on binary, small multiclass and high-dimensional tasks.
* **SVM and KNN** (with scaling) generally perform strongly on clean numeric data, and SVM is typically the most reliable overall, with low log loss thanks to calibrated probabilities.
* **Decision Trees** are the most interpretable but are usually less accurate and give over-confident probabilities (higher log loss) unless depth is controlled or ensembles are used.
* **Naïve Bayes** is the fastest and needs the least data, but its independence assumption hurts when features are correlated (e.g. image pixels), so it tends to rank lower there.
* **Accuracy alone is not enough** – precision/recall matter when error costs differ (e.g. cancer screening), and log loss reveals how trustworthy the predicted probabilities are.
* **Recommendation:** choose the algorithm by the task – SVM for accuracy on medium-sized/high-dimensional data, Decision Tree when explainability is required, KNN for simple low-dimensional problems, and Naïve Bayes for speed and small-data or text problems. Always scale features where needed, cross-validate and tune hyper-parameters.